<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/BTC_1_min_predict_next_day.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install yfinance

import yfinance as yf
import pandas as pd

# Загрузка данных Bitcoin в реальном времени
# Ticker для Bitcoin на Yahoo Finance: BTC-USD
ticker = 'BTC-USD'

# Загружаем данные (например, за последние 2 года с интервалом 1 день)
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Сбрасываем индекс, чтобы Date стала колонкой
    df = df.reset_index()

    # Приведение даты к формату YYYY-MM-DD
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

    # Yahoo Finance API через библиотеку yfinance сразу возвращает нужные колонки:
    # Date, Open, High, Low, Close, Adj Close, Volume

    print(f'Данные {ticker} успешно загружены напрямую из Yahoo Finance.')
    display(df.head())

    # Сохраняем локально в сессионное хранилище Colab (не на диск)
    df.to_csv('live_yahoo_btc.csv', index=False)
else:
    print('Не удалось получить данные. Проверьте подключение к интернету.')

/tmp/ipykernel_21390/3755746512.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Данные BTC-USD успешно загружены напрямую из Yahoo Finance.


Price,Date,Close,High,Low,Open,Volume
Ticker,,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
0,2024-05-06,63161.949219,65494.902344,62746.238281,64038.312500,28697928697
1,2024-05-07,62334.816406,64390.457031,62285.980469,63162.761719,25930730982
2,2024-05-08,61187.941406,62986.085938,60877.128906,62332.640625,26088172222
3,2024-05-09,63049.960938,63404.914062,60648.074219,61191.199219,25453338161
4,2024-05-10,60792.777344,63446.742188,60208.781250,63055.191406,27804954694


In [29]:
from sklearn.linear_model import LinearRegression

# Train machine learning model using fetched 'df' from Yahoo Finance
# Ensure data exists and prepare features
X = df['Open'].values.reshape(-1, 1)
y = df['High'].values.reshape(-1, 1)

model = LinearRegression().fit(X, y)
print("Model trained successfully on Yahoo Finance data.")

Model trained successfully on Yahoo Finance data.


In [30]:
# Using .iloc[0] to avoid FutureWarning when converting single-element series to float
current_price_val = float(df.iloc[-1]["Open"].iloc[0])
predicted_price = model.predict([[current_price_val]])[0][0]

print(f"Current Open Price: {current_price_val}")
print(f"Predicted High Price: {predicted_price}")

# Compare the predicted high price with the current Bitcoin price every second for the next hour.
for i in range(3600):
    # Accessing the scalar value directly to avoid warnings
    current_price_check = float(df.iloc[-1]["Open"].iloc[0])
    if current_price_check >= predicted_price:
        print("Sell order executed at price:", current_price_check)
        break
    elif i == 3599:
        print("Sell order executed due to model inefficiency at price:", current_price_check)
    # time.sleep(1)

Current Open Price: 80900.7734375
Predicted High Price: 82363.38083943573
Sell order executed due to model inefficiency at price: 80900.7734375


In [31]:
###############################################################################

In [32]:
###############################################################################

In [33]:
#VERSION 2
from sklearn import linear_model

# Using 'df' which contains the Yahoo Finance data instead of the empty 'merged_data'
train_df = df.copy()
train_df.dropna(inplace=True) # remove rows with missing values

# Standardize features and target based on available columns
X_train = train_df[["Open"]].values
y_train = train_df[["High"]].values

model = linear_model.LinearRegression()
model.fit(X_train, y_train)
print("Model trained successfully on 'df' data.")

Model trained successfully on 'df' data.


In [34]:
# Use the prediction algorithm to predict the high price based on the last open price
# Using .iloc[0] to handle the yfinance Series structure
current_price_val = float(df.iloc[-1]["Open"].iloc[0])
predicted_price = model.predict([[current_price_val]])[0][0]

print(f"Current Price: {current_price_val}")
print(f"Predicted High: {predicted_price}")

# Compare the predicted high price with the current Bitcoin price every second for the next hour.
for i in range(3600):
    # In this simulation, we check against the latest available price in our dataframe
    check_price = float(df.iloc[-1]["Open"].iloc[0])
    if check_price >= predicted_price:
        print("Sell order executed at price:", check_price)
        break
    elif i == 3599:
        print("Sell order executed due to model inefficiency at price:", check_price)
    # time.sleep(1)

Current Price: 80900.7734375
Predicted High: 82363.38083943573
Sell order executed due to model inefficiency at price: 80900.7734375


In [35]:
###############################################################################

In [36]:
###############################################################################

In [37]:
# VERSION 3

In [38]:
from sklearn import linear_model
import pandas as pd

# select features and target using 'df' which contains Yahoo Finance data
X_train = df[["Open"]].values
y_train = df[["High"]].values

# create linear regression model
model = linear_model.LinearRegression()

# fit model to the data
model.fit(X_train, y_train)

# predict using the model
# Accessing the last 'Open' price as a scalar
current_price_val = float(df.iloc[-1]["Open"].iloc[0])
# Reshaping input to 2D array as expected by sklearn
predicted_price = model.predict([[current_price_val]])[0][0]

print(f"Model trained and prediction made.")
print(f"Current Open: {current_price_val}, Predicted High: {predicted_price}")

Model trained and prediction made.
Current Open: 80900.7734375, Predicted High: 82363.38083943573


In [67]:
# Simulation with Stop Loss
stop_loss_pct = 0.02 # 2% loss limit

# Safely extract the last Open price
# We check if it's a MultiIndex or single index to get the scalar value correctly
if isinstance(df.columns, pd.MultiIndex):
    entry_price = float(df["Open"].iloc[-1].iloc[0])
else:
    entry_price = float(df["Open"].iloc[-1])

stop_loss_price = entry_price * (1 - stop_loss_pct)

print(f"Entry Price: {entry_price}")
print(f"Target High: {predicted_price}")
print(f"Stop Loss: {stop_loss_price}")

for i in range(3600):
    # Using the latest value from the dataframe
    if isinstance(df.columns, pd.MultiIndex):
        current_price = float(df["Open"].iloc[-1].iloc[0])
    else:
        current_price = float(df["Open"].iloc[-1])

    if current_price >= predicted_price:
        print(f"Target reached! Sell order executed at profit: {current_price}")
        break
    elif current_price <= stop_loss_price:
        print(f"Stop Loss triggered! Sell order executed to limit loss at: {current_price}")
        break
    elif i == 3599:
        print(f"Time limit reached. Sell order executed due to model inefficiency at: {current_price}")

Entry Price: 80900.7734375
Target High: 82363.38083943573
Stop Loss: 79282.75796875
Time limit reached. Sell order executed due to model inefficiency at: 80900.7734375


In [40]:
###############################################################################

In [41]:
###############################################################################

In [42]:
# OTHER

In [43]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import datetime as dt

# Using the live 'df' from Yahoo Finance instead of 'merged_data'
train_data = df.copy()

# Get column names. We use level 0 because yfinance returns a MultiIndex.
all_cols = [col[0] if isinstance(col, tuple) else col for col in train_data.columns]
train_data.columns = all_cols

# Define features and target
features = [col for col in train_data.columns if col not in ['High', 'Date']]
X_train = train_data[['Open']].values
y_train = train_data['High'].values

model = LinearRegression()

# Fit the model using the scalar values from 'Open'
model.fit(X_train, y_train)

# Prepare test data (using the same 'df' for demonstration)
X_test = train_data[['Open']].values
y_pred = model.predict(X_test)

print("Model trained and predictions made successfully.")

Model trained and predictions made successfully.


In [44]:
#TEMP
# import pandas as pd
# from sklearn.linear_model import LinearRegression

# # Load the training dataset and fit the LinearRegression model
# train_data = pd.read_csv("train.csv")
# X_train = train_data[["feature1", "feature2"]]
# y_train = train_data["target"]
# model = LinearRegression()
# model.fit(X_train, y_train)

# # Load the testing dataset and set the column names to match the training dataset
# test_data = pd.read_csv("test.csv")
# test_data.columns = ["feature1", "feature2"]

# # Use the trained model to make predictions on the testing dataset
# X_test = test_data[["feature1", "feature2"]]
# y_pred = model.predict(X_test)

In [45]:
#1 Hour

In [47]:
import pandas as pd

# Вместо чтения файла с Google Drive, используем данные 'df', загруженные ранее из Yahoo Finance
if 'df' in globals() and not df.empty:
    data = df.copy()

    # Если Yahoo Finance вернул MultiIndex, упрощаем его
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [col[0] for col in data.columns]

    # Выполняем аналогичную очистку, если необходимо
    if "tradecount" in data.columns:
        data = data.drop(columns=["tradecount"])

    print("Данные успешно скопированы из основного датафрейма.")
    display(data.head())
else:
    print("Ошибка: Основной датафрейм 'df' пуст или не существует. Пожалуйста, запустите первую ячейку.")

Данные успешно скопированы из основного датафрейма.


,Date,Close,High,Low,Open,Volume
0,2024-05-06,63161.949219,65494.902344,62746.238281,64038.312500,28697928697
1,2024-05-07,62334.816406,64390.457031,62285.980469,63162.761719,25930730982
2,2024-05-08,61187.941406,62986.085938,60877.128906,62332.640625,26088172222
3,2024-05-09,63049.960938,63404.914062,60648.074219,61191.199219,25453338161
4,2024-05-10,60792.777344,63446.742188,60208.781250,63055.191406,27804954694


In [49]:
from sklearn.linear_model import LinearRegression

# Train machine learning model using the live 'df' data
# We ensure to reshape the values into the 2D array format required by scikit-learn
X = df['Open'].values.reshape(-1, 1)
y = df['High'].values.reshape(-1, 1)

model = LinearRegression().fit(X, y)
print("Model trained successfully using live Yahoo Finance data.")

Model trained successfully using live Yahoo Finance data.


In [68]:
# Simulation with Stop Loss (version 2)
stop_loss_pct = 0.02

if isinstance(df.columns, pd.MultiIndex):
    entry_price = float(df["Open"].iloc[-1].iloc[0])
else:
    entry_price = float(df["Open"].iloc[-1])

stop_loss_price = entry_price * (1 - stop_loss_pct)

for i in range(3600):
    if isinstance(df.columns, pd.MultiIndex):
        current_price = float(df["Open"].iloc[-1].iloc[0])
    else:
        current_price = float(df["Open"].iloc[-1])

    if current_price >= predicted_price:
        print("Target reached! Sell order executed at price:", current_price)
        break
    elif current_price <= stop_loss_price:
        print("STOP LOSS TRIGGERED at price:", current_price)
        break
    elif i == 3599:
        print("Sell order executed due to model inefficiency at price:", current_price)

Sell order executed due to model inefficiency at price: 80900.7734375


In [ ]:
import pandas as pd
import numpy as np

# Load historical data and latest price
historical_data = pd.read_csv("/content/drive/MyDrive/Bitcoin-Historical-Dataset (31.03.2023)/merged_data.csv")
historical_data['Close'] = historical_data['Close'].astype(int)
latest_price = historical_data["Close"].iloc[-1]

# Find the index of the latest price in the historical data
latest_index = np.where(historical_data["Close"].values == latest_price)[0][-1]

# Count how many times the latest price appeared in the historical data and when was it
count_latest_price = np.sum(historical_data["Close"].values == latest_price)
latest_price_dates = historical_data.loc[historical_data["Close"].values == latest_price, "Date"].values

# Find the indices of the 3rd prices after the latest price in the historical data
third_high_index = np.where(historical_data["Close"].shift(-3) > historical_data["Close"].shift(-1))[0]
third_low_index = np.where(historical_data["Close"].shift(-3) < historical_data["Close"].shift(-1))[0]

# Count how many times the 3rd price after the latest price was higher and lower, respectively
count_third_high = len(np.intersect1d(third_high_index, latest_index+3))
count_third_low = len(np.intersect1d(third_low_index, latest_index+3))

# Print the results
print(f"The latest price {latest_price} appeared {count_latest_price} times in the historical data on the following dates: {latest_price_dates}")
print(f"The 3rd price after the latest price was higher {count_third_high} times and lower {count_third_low} times.")



In [ ]:
import pandas as pd

# Load historical data and latest price
historical_data = pd.read_csv("/content/drive/MyDrive/Bitcoin-Historical-Dataset (31.03.2023)/merged_data.csv")
historical_data['Close'] = historical_data['Close'].astype(int)
latest_price = historical_data["Close"].iloc[-1]

# Convert the Close column to integers
historical_data["Close"] = historical_data["Close"].astype(int)

# Find the index of the latest price in the historical data
latest_index = historical_data.index[0]

# Find the index of the 3rd price after the latest price in the historical data
third_price_index = latest_index + 3

# Get the latest and 3rd price after the latest price
latest_price_close = historical_data.iloc[latest_index]["Close"]
third_price_close = historical_data.iloc[third_price_index]["Close"]

# Initialize counters for higher and lower 3rd prices
higher_count = 0
lower_count = 0

# Loop through the historical data to count higher and lower 3rd prices
for i in range(latest_index + 4, len(historical_data)):
    third_price = historical_data.iloc[i]["Close"]
    if third_price > third_price_close:
        higher_count += 1
    elif third_price < third_price_close:
        lower_count += 1

# Print the results
if latest_price_close > third_price_close:
    print(f"The latest close is greater than the 3rd close after the latest close {higher_count} times and less than the 3rd close {lower_count} times.")
elif latest_price_close < third_price_close:
    print(f"The latest close is less than the 3rd close after the latest close {lower_count} times and greater than the 3rd close {higher_count} times.")
else:
    print("The latest close is equal to the 3rd close after the latest close.")


In [64]:
import yfinance as yf
import pandas as pd
import numpy as np

# Load historical data and latest price
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float and then int for the logic below
    merged_data['Close'] = merged_data['Close'].astype(float)
    merged_data['Close_Int'] = merged_data['Close'].astype(int)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()

    latest_price = merged_data["Close_Int"].iloc[-1]

    # Find the index of the latest price in the historical data
    latest_index = np.where(merged_data["Close_Int"] == latest_price)[0][-1]

    # Find the indices where the 3rd price after a specific price was higher/lower
    # We shift -3 to look ahead and compare it to the price at shift -1
    third_high_index = np.where(merged_data["Close_Int"].shift(-3) > merged_data["Close_Int"].shift(-1))[0]
    third_low_index = np.where(merged_data["Close_Int"].shift(-3) < merged_data["Close_Int"].shift(-1))[0]

    # Count occurrences relative to the current latest index position logic
    count_third_high = len(np.intersect1d(third_high_index, latest_index - 3))
    count_third_low = len(np.intersect1d(third_low_index, latest_index - 3))

    # Print the results
    # Comparing latest price to the price 3 days ago (iloc[-4])
    if len(merged_data) >= 4:
        reference_price = merged_data["Close_Int"].iloc[-4]
        if latest_price > reference_price:
            print(f"The latest price {latest_price} is higher than the price 3 days ago ({reference_price}).")
        elif latest_price < reference_price:
            print(f"The latest price {latest_price} is lower than the price 3 days ago ({reference_price}).")

        print(f"In history, when at this relative position, the 3rd price after was higher {count_third_high} times and lower {count_third_low} times.")
else:
    print("Failed to download data.")

/tmp/ipykernel_21390/666751052.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

The latest price 82361 is higher than the price 3 days ago (78538).
In history, when at this relative position, the 3rd price after was higher 1 times and lower 0 times.


### Анализ прибыльности проекта (Bitcoin Prediction & Trading Bot)

На основе текущего кода можно выделить несколько ключевых факторов, влияющих на потенциальную прибыльность:

#### 1. Стратегия регрессии (Open → High)
*   **Идея:** Предсказать максимальную цену дня на основе цены открытия.
*   **Прибыльность:** В теории это позволяет выставить `Take Profit` на уровне `Predicted High`.
*   **Проблема:** Линейная регрессия на одном признаке (`Open`) слишком проста. Она предсказывает среднее историческое отклонение, но не учитывает рыночную волатильность и новости.

#### 2. Симуляция торговли (Cell `sXwF9f9raeSS`)
*   **Логика:** Бот ждет 1 час. Если цена не достигла цели, он продает в минус или «в ноль».
*   **Риск:** За этот час цена может упасть на 5-10%, и бот зафиксирует убыток. Отсутствие `Stop Loss` — главная угроза капиталу.

#### 3. Статистический анализ (Cell `4Wg9aGLsA7UM`)
*   **Результат:** Вы нашли, что в похожих ситуациях цена росла в 100% случаев (1 из 1).
*   **Вывод:** Выборка (n=1) слишком мала для принятия финансовых решений. Нужно расширить период данных с 2 лет до 5-10 лет для подтверждения гипотезы.

#### 4. Комиссии и проскальзывание
*   Код не учитывает комиссии биржи (обычно 0.1% за сделку). Если предсказанная прибыль меньше 0.2-0.3%, сделка будет убыточной даже при верном прогнозе.

### Итоговая оценка:
**Текущая прибыльность: Низкая / Рискованная.**
Проект является отличным «каркасом», но для реальной прибыли необходимо:
1.  **Усложнить модель:** Добавить индикаторы (RSI, MACD, Volume) в `X_train`.
2.  **Добавить Stop Loss:** Обязательная продажа при падении цены на N%.
3.  **Бэктестинг:** Прогнать алгоритм на исторических данных за весь 2024-2025 год и посчитать итоговый баланс.

In [62]:
import pandas as pd

# Load the merged data and convert the Close column to integer
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()
merged_data['Close'] = merged_data['Close'].astype(int)

# Find the dates where the latest close price occurred
latest_close = merged_data['Close'].iloc[-1]
latest_close_dates = merged_data[merged_data['Close'] == latest_close]['Date'].values

# Find the next candle after the latest close price for each date
buy_count = 0
sell_count = 0
for date in latest_close_dates:
    next_candle = merged_data.loc[merged_data['Date'] == date, 'Close'].shift(-1)
    if pd.isna(next_candle.iloc[0]):
        continue

   # Calculate the buy/sell value
    if next_candle.iloc[0] > latest_close:
        buy_sell = 'buy'
        value = next_candle.iloc[0] - latest_close
        buy_count += 1

    else:
        buy_sell = 'sell'
        value = latest_close - next_candle.iloc[0]
        sell_count += 1

    # Print the result
    print(f"Latest close price of {latest_close} occurred on {date}.")
    print(f"The next candle after {date} was {buy_sell}.\n")

# Print the buy/sell counts and the STRONG BUY/SELL statement
print(f"Buy signals: {buy_count}")
print(f"Sell signals: {sell_count}")
if buy_count > sell_count:
    print("STRONG BUY")
else:
    print("STRONG SELL")

/tmp/ipykernel_21390/3129952145.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Buy signals: 0
Sell signals: 0
STRONG SELL


In [61]:
import pandas as pd

# Load the merged data and convert the Close column to integer
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()
merged_data['Close'] = merged_data['Close'].astype(int)

# Find the dates where the latest close price occurred
latest_close = merged_data['Close'].iloc[-1]
latest_close_dates = merged_data[merged_data['Close'] == latest_close]['Date'].values

# Find the next 3rd candle after the latest close price for each date
for date in latest_close_dates:
    next_candle = merged_data.loc[merged_data['Date'] == date, 'Close'].shift(-1)
    if pd.isna(next_candle.iloc[0]):
        continue
    next_2_candle = merged_data.loc[merged_data['Date'] == date, 'Close'].shift(-2)
    if pd.isna(next_2_candle.iloc[0]):
        continue
    next_3_candle = merged_data.loc[merged_data['Date'] == date, 'Close'].shift(-3)
    if pd.isna(next_3_candle.iloc[0]):
        continue

    # Calculate the buy/sell value
    if next_candle.iloc[0] > latest_close and next_2_candle.iloc[0] > latest_close and next_3_candle.iloc[0] > latest_close:
        buy_sell = 'buy'
        value = next_candle.iloc[0] - latest_close
    else:
        next_candle.iloc[0] < latest_close and next_2_candle.iloc[0] < latest_close and next_3_candle.iloc[0] < latest_close
        buy_sell = 'sell'
        value = latest_close - next_candle.iloc[0]

    # Print the result
    print(f"Latest close price of {latest_close} occurred on {date}.")
    print(f"The 3rd next candle after {date} was {buy_sell} with a value of {value}.")


/tmp/ipykernel_21390/1726297835.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed


In [60]:
import pandas as pd

# Load the merged data and convert the Close column to float
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()
merged_data['Close'] = merged_data['Close'].astype(float)

# Calculate the percentage change from the previous close price
merged_data['Pct_Change'] = merged_data['Close'].pct_change()

# Find the dates where the percentage change was greater than 5%
change_threshold = 0.05
change_dates = merged_data[(merged_data['Pct_Change'] > change_threshold) | (merged_data['Pct_Change'] < -change_threshold)]['Date'].values

# Find the weekday and time for each date where the percentage change was greater than 5%
for date in change_dates:
    # Find the index of the current date
    date_index = merged_data[merged_data['Date'] == date].index[0]

    # Calculate the percentage change from the previous close price
    pct_change = merged_data.loc[date_index, 'Pct_Change']

    # Calculate the time and weekday of the current date
    timestamp = pd.to_datetime(date).to_pydatetime()
    time = timestamp.time().strftime('%H:%M:%S')
    weekday = timestamp.strftime('%A')

    # Print the result
    if pct_change > 0:
        print(f"Close price went up more than {change_threshold*100:.2f}% on {weekday} at {time}.")
    else:
        print(f"Close price went down more than {change_threshold*100:.2f}% on {weekday} at {time}.")


/tmp/ipykernel_21390/1460771625.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Close price went up more than 5.00% on Wednesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Thursday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Friday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went up more than 5.00% on Thursday at 00:00:00.
Close price went up more than 5.00% on Friday at 00:00:00.
Close price went down more than 5.00% on Tuesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went up more than 5.00% on Wednesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Wednesday at 00:00:00.
Close price went down more than 5.00% on Tuesday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went up more 

In [58]:
import pandas as pd

# Load the merged data and convert the Close column to float
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()
merged_data['Close'] = merged_data['Close'].astype(float)

# Calculate the percentage change from the previous close price
merged_data['Pct_Change'] = merged_data['Close'].pct_change()

# Find the dates where the percentage change was greater than 5%
change_threshold = 0.05
change_dates = merged_data[(merged_data['Pct_Change'] > change_threshold) | (merged_data['Pct_Change'] < -change_threshold)]['Date'].values

# Find the weekday and time for each date where the percentage change was greater than 5%
day_count = {}
hour_count = {}
for date in change_dates:
    # Find the index of the current date
    date_index = merged_data[merged_data['Date'] == date].index[0]

    # Calculate the percentage change from the previous close price
    pct_change = merged_data.loc[date_index, 'Pct_Change']

    # Calculate the time and weekday of the current date
    timestamp = pd.to_datetime(date).to_pydatetime()
    time = timestamp.time().strftime('%H:%M:%S')
    weekday = timestamp.strftime('%A')

    # Increment the day count for the current weekday
    if weekday in day_count:
        day_count[weekday] += 1
    else:
        day_count[weekday] = 1

    # Increment the hour count for the current hour
    if time in hour_count:
        hour_count[time] += 1
    else:
        hour_count[time] = 1

    # Print the result
    if pct_change > 0:
        print(f"Close price went up more than {change_threshold*100:.2f}% on {weekday} at {time}.")
    else:
        print(f"Close price went down more than {change_threshold*100:.2f}% on {weekday} at {time}.")

# Print the day and hour counts
print("Day counts:")
for day, count in day_count.items():
    print(f"{day}: {count}")

print("Hour counts:")
for hour, count in hour_count.items():
    print(f"{hour}: {count}")


/tmp/ipykernel_21390/3788602063.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Close price went up more than 5.00% on Wednesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Thursday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Friday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went up more than 5.00% on Thursday at 00:00:00.
Close price went up more than 5.00% on Friday at 00:00:00.
Close price went down more than 5.00% on Tuesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went up more than 5.00% on Wednesday at 00:00:00.
Close price went up more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went down more than 5.00% on Wednesday at 00:00:00.
Close price went down more than 5.00% on Tuesday at 00:00:00.
Close price went down more than 5.00% on Monday at 00:00:00.
Close price went up more 

In [56]:
import pandas as pd

# Load the merged data and convert the Close column to float
ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()

merged_data['Close'] = merged_data['Close'].astype(float)

# Calculate the percentage change from the previous close price
merged_data['Pct_Change'] = merged_data['Close'].pct_change()

# Find the dates where the percentage change was greater than 5%
change_threshold = 0.05
change_dates = merged_data[(merged_data['Pct_Change'] > change_threshold) | (merged_data['Pct_Change'] < -change_threshold)]['Date'].values

# Filter the change dates to only include Fridays
friday_dates = [date for date in change_dates if pd.to_datetime(date).weekday() == 4]

# Count the frequency of each hour in which the price change occurred on Fridays
hour_count = {}
for date in friday_dates:
    # Find the index of the current date
    date_index = merged_data[merged_data['Date'] == date].index[0]

    # Calculate the percentage change from the previous close price
    pct_change = merged_data.loc[date_index, 'Pct_Change']

    # Calculate the hour of the current date
    timestamp = pd.to_datetime(date).to_pydatetime()
    hour = timestamp.time().hour

    # Increment the count for the current hour
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

# Print the hour(s) with the highest frequency of price change on Fridays
max_count = max(hour_count.values())
print("On Fridays, the price went up or down more than 5% most frequently at the following hour(s):")
for hour, count in hour_count.items():
    if count == max_count:
        print(f"{hour}:00")


/tmp/ipykernel_21390/3714654363.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

On Fridays, the price went up or down more than 5% most frequently at the following hour(s):
0:00


In [55]:
import pandas as pd

ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()

# Calculate the percentage change from the previous close price
merged_data['Pct_Change'] = merged_data['Close'].pct_change()

# Find the dates where the percentage change was greater than 5%
change_threshold = 0.05
change_dates = merged_data[(merged_data['Pct_Change'] > change_threshold) | (merged_data['Pct_Change'] < -change_threshold)]['Date'].values

# Find the weekday and time for each date where the percentage change was greater than 5%
for date in change_dates:
    # Find the index of the current date
    date_index = merged_data[merged_data['Date'] == date].index[0]

    # Calculate the percentage change from the previous close price
    pct_change = merged_data.loc[date_index, 'Pct_Change']

    # Calculate the time and weekday of the current date
    timestamp = pd.to_datetime(date).to_pydatetime()
    time = timestamp.time().strftime('%H:%M:%S')
    weekday = timestamp.strftime('%A')

    # Print the result
    if weekday == 'Friday':
        if pct_change > 0:
            print(f"Close price went up by more than {change_threshold*100:.2f}% on {weekday} at {time}.")
        else:
            print(f"Close price went down by more than {change_threshold*100:.2f}% on {weekday} at {time}.")


/tmp/ipykernel_21390/2361938404.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Close price went down by more than 5.00% on Friday at 00:00:00.
Close price went up by more than 5.00% on Friday at 00:00:00.
Close price went down by more than 5.00% on Friday at 00:00:00.
Close price went down by more than 5.00% on Friday at 00:00:00.
Close price went up by more than 5.00% on Friday at 00:00:00.


In [53]:
import pandas as pd
import yfinance as yf

ticker = 'BTC-USD'

# Download data
df = yf.download(ticker, period='2y', interval='1d')

if not df.empty:
    # Flatten MultiIndex if necessary
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Reset index to make Date a column
    merged_data = df.reset_index().copy()

    # Ensure Close is float
    merged_data['Close'] = merged_data['Close'].astype(float)

    # Convert Date to string format for comparison
    merged_data['Date'] = merged_data['Date'].dt.strftime('%Y-%m-%d')

    # Calculate the percentage change from the previous close price
    merged_data['Pct_Change'] = merged_data['Close'].pct_change()

    # Find the dates where the percentage change was greater than 5%
    change_threshold = 0.05
    change_mask = (merged_data['Pct_Change'] > change_threshold) | (merged_data['Pct_Change'] < -change_threshold)
    change_dates = merged_data[change_mask]['Date'].values

    # Find the weekday and time for each date where change was > 5% on Fridays
    friday_change_count = {'up': 0, 'down': 0}
    for date in change_dates:
        # Get the row for this date
        row = merged_data[merged_data['Date'] == date].iloc[0]
        pct_change = row['Pct_Change']

        # Check if the date is a Friday
        timestamp = pd.to_datetime(date)
        if timestamp.weekday() != 4:  # 4 corresponds to Friday
            continue

        # Increment the count for up or down changes on Fridays
        if pct_change > 0:
            friday_change_count['up'] += 1
            direction = "up"
        else:
            friday_change_count['down'] += 1
            direction = "down"

        print(f"Close price went {direction} more than {change_threshold*100:.2f}% on Friday ({date}).")

    print(f"\nSummary:")
    print(f"There were {friday_change_count['up']} Fridays where the price went up more than {change_threshold*100:.2f}%")
    print(f"There were {friday_change_count['down']} Fridays where the price went down more than {change_threshold*100:.2f}%")
else:
    print("No data downloaded.")

/tmp/ipykernel_21390/2244113688.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period='2y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Close price went down more than 5.00% on Friday (2024-08-02).
Close price went up more than 5.00% on Friday (2024-08-23).
Close price went down more than 5.00% on Friday (2025-10-10).
Close price went down more than 5.00% on Friday (2025-11-14).
Close price went up more than 5.00% on Friday (2026-02-06).

Summary:
There were 2 Fridays where the price went up more than 5.00%
There were 3 Fridays where the price went down more than 5.00%
